# AgroSense — Entrenamiento del modelo de cultivos (Colab)

Pipeline de prueba de concepto para el módulo de **cultivos** de AgroSense: clasificación de
enfermedades en hojas de **maíz, tomate, papa y café** vía transfer learning con MobileNetV2,
exportado a `.tflite` para integrarse en la app Kotlin (carpeta `assets/`).

**Meta de estas 1-2 semanas:** validar el componente más incierto del proyecto de forma aislada,
antes de que exista ninguna pantalla de la app.

Flujo de este notebook:
1. Setup y montaje de Drive
2. Descarga del dataset (PlantVillage + nota sobre café)
3. Filtrado a los 4 cultivos MVP
4. Pipeline de datos (`tf.data`, augmentation)
5. Modelo: MobileNetV2 — fase *freeze* → fase *fine-tune*
6. Evaluación con split de prueba
7. Validación con PlantDoc (fotos de campo real)
8. Exportar `.tflite` cuantizado + `labels.txt`

> ⚠️ **Nota sobre el dataset de café:** el PlantVillage "clásico" (el que se usa vía Kaggle/TFDS)
> cubre maíz, tomate y papa, pero **no incluye café**. Este notebook está armado para que cafeto
> se agregue como una carpeta adicional dentro de la misma estructura de carpetas
> (`data/raw/coffee/<clase>/*.jpg`), usando un dataset separado de hojas de café (por ejemplo
> RoCoLe u otro dataset de roya del café en Kaggle/Roboflow). Revisa la licencia de ese dataset
> antes de usarlo, igual que se documentó para los datasets ganaderos.


## 1. Setup

In [ ]:
# Librerías. TensorFlow ya viene preinstalado en Colab.
import os
import pathlib
import shutil
import random
import json

import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models

print("TensorFlow:", tf.__version__)
print("GPU disponible:", tf.config.list_physical_devices('GPU'))


In [ ]:
# Montar Google Drive — aquí vivirá el dataset y los checkpoints/exports,
# para no perder trabajo entre sesiones de Colab.
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/AgroSense'
DATA_DIR = os.path.join(PROJECT_DIR, 'data')
RAW_DIR = os.path.join(DATA_DIR, 'raw')          # dataset completo tal como se descarga
MVP_DIR = os.path.join(DATA_DIR, 'mvp')          # solo las clases de los 4 cultivos MVP
PLANTDOC_DIR = os.path.join(DATA_DIR, 'plantdoc')  # fotos de campo real, para validación
EXPORT_DIR = os.path.join(PROJECT_DIR, 'export')
CKPT_DIR = os.path.join(PROJECT_DIR, 'checkpoints')

for d in [DATA_DIR, RAW_DIR, MVP_DIR, PLANTDOC_DIR, EXPORT_DIR, CKPT_DIR]:
    os.makedirs(d, exist_ok=True)

print("Proyecto en Drive:", PROJECT_DIR)


## 2. Descargar el dataset

Usamos `kagglehub` para bajar PlantVillage directo a Colab (no requiere subir manualmente nada
a Drive). Necesitas una cuenta de Kaggle y tu `kaggle.json` (Kaggle → Account → Create New API
Token). Colab te va a pedir subirlo la primera vez.


In [ ]:
!pip install -q kagglehub

import kagglehub

# Dataset con las 38 clases (14 especies) en formato "color" — incluye maíz, tomate y papa.
plantvillage_path = kagglehub.dataset_download("abdallahalidev/plantvillage-dataset")
print("PlantVillage descargado en:", plantvillage_path)

# La estructura típica de este dataset es:
#   <plantvillage_path>/plantvillage dataset/color/<Especie>___<clase>/*.jpg
# Ajusta PV_COLOR_DIR si kagglehub lo descomprime distinto — imprime el árbol para confirmar.
import subprocess
print(subprocess.run(['find', plantvillage_path, '-maxdepth', '3'], capture_output=True, text=True).stdout[:2000])


In [ ]:
# Copiamos el dataset crudo a Drive UNA sola vez (para no re-descargarlo cada sesión).
# Si ya existe en Drive, este paso se salta.
PV_COLOR_DIR_CANDIDATES = [
    os.path.join(plantvillage_path, 'plantvillage dataset', 'color'),
    os.path.join(plantvillage_path, 'plantvillage-dataset', 'color'),
]
PV_COLOR_DIR = next((p for p in PV_COLOR_DIR_CANDIDATES if os.path.isdir(p)), None)
assert PV_COLOR_DIR, "No se encontró la carpeta 'color' del dataset — revisa el print del árbol arriba y ajusta la ruta."

target = os.path.join(RAW_DIR, 'plantvillage_color')
if not os.path.isdir(target):
    print("Copiando a Drive (una sola vez, puede tardar unos minutos)...")
    shutil.copytree(PV_COLOR_DIR, target)
else:
    print("Ya existe en Drive, se omite la copia:", target)

print(sorted(os.listdir(target))[:10], "...")


### 2.1 Café (dataset complementario)

PlantVillage no incluye café. Sube o descarga tu dataset de café dentro de
`data/raw/coffee_raw/<clase>/*.jpg` en Drive (por ejemplo un dataset de roya/mancha de hoja de
café). La celda siguiente solo verifica que la carpeta exista antes de seguir — si no la tienes
lista todavía, puedes correr el resto del notebook solo con maíz/tomate/papa y agregar café
después sin rehacer nada.


In [ ]:
COFFEE_RAW_DIR = os.path.join(RAW_DIR, 'coffee_raw')
INCLUDE_COFFEE = os.path.isdir(COFFEE_RAW_DIR) and len(os.listdir(COFFEE_RAW_DIR)) > 0

if INCLUDE_COFFEE:
    print("Café encontrado, se incluirá en el MVP:", os.listdir(COFFEE_RAW_DIR))
else:
    print("No se encontró dataset de café en", COFFEE_RAW_DIR)
    print("Continuamos solo con maíz, tomate y papa. Puedes agregar café más adelante y re-correr desde la sección 3.")


## 3. Filtrar a los 4 cultivos MVP

Del set completo de 38 clases nos quedamos solo con las carpetas de **maíz (Corn), tomate
(Tomato) y papa (Potato)**, más café si está disponible. Esto reduce el tiempo de entrenamiento y
enfoca el modelo en el alcance real acordado para el MVP.


In [ ]:
MVP_CROP_KEYWORDS = ['corn', 'maize', 'tomato', 'potato']

all_classes = sorted(os.listdir(target))
mvp_classes = [c for c in all_classes if any(k in c.lower() for k in MVP_CROP_KEYWORDS)]

# ---------------------------------------------------------------------------
# PUNTO DE DECISIÓN — léelo antes de seguir.
#
# El filtro por palabras clave se queda con TODAS las clases de maíz, tomate y
# papa que trae PlantVillage: normalmente 17 (4 de maíz, 3 de papa, 10 de
# tomate), no 10. Y con los nombres crudos del dataset, tipo
# "Corn_(maize)___Common_rust_", que no son los nombres "limpios" que aparecen
# en la documentación de la app.
#
# Esa lista define las ETIQUETAS DEL MODELO y su ORDEN. La app resuelve el
# nombre de cada predicción por índice contra labels.txt, así que este orden es
# un contrato: si luego no coincide, los diagnósticos salen con el nombre
# equivocado y sin ningún error visible.
#
# Dos opciones:
#
#  (a) Quedarte con las 17. Más enfermedades distinguidas, más realista.
#      No tienes que hacer nada: deja CLASES_ELEGIDAS = None.
#
#  (b) Recortar a un subconjunto. Copia los nombres EXACTOS que imprime la
#      celda abajo dentro de CLASES_ELEGIDAS. Tiene que ser aquí, antes de que
#      se cree MVP_DIR, porque las etiquetas salen de los nombres de carpeta.
#
# NUNCA escribas labels.txt a mano. Lo genera la sección 8 con el orden real.
# ---------------------------------------------------------------------------

CLASES_ELEGIDAS = None  # o una lista de nombres exactos, p. ej. ['Potato___healthy', ...]

if CLASES_ELEGIDAS:
    desconocidas = set(CLASES_ELEGIDAS) - set(mvp_classes)
    assert not desconocidas, f"Estos nombres no existen en el dataset: {sorted(desconocidas)}"
    mvp_classes = [c for c in mvp_classes if c in CLASES_ELEGIDAS]

print(f"{len(mvp_classes)} clases seleccionadas de {len(all_classes)} totales:")
for c in mvp_classes:
    print(" -", c)


In [ ]:
# Copiamos (symlink es más rápido que copiar, pero usamos copy para que sea robusto
# si luego subes esto a otra máquina/entorno).
if os.path.isdir(MVP_DIR) and len(os.listdir(MVP_DIR)) > 0:
    print("MVP_DIR ya tiene contenido, se omite para no duplicar. Bórralo manualmente si quieres regenerar.")
else:
    for c in mvp_classes:
        src = os.path.join(target, c)
        dst = os.path.join(MVP_DIR, c)
        shutil.copytree(src, dst)

    if INCLUDE_COFFEE:
        for c in os.listdir(COFFEE_RAW_DIR):
            src = os.path.join(COFFEE_RAW_DIR, c)
            dst = os.path.join(MVP_DIR, f"Coffee___{c}")
            if os.path.isdir(src):
                shutil.copytree(src, dst)

final_classes = sorted(os.listdir(MVP_DIR))
print(f"\nMVP_DIR listo con {len(final_classes)} clases:")
for c in final_classes:
    n = len(os.listdir(os.path.join(MVP_DIR, c)))
    print(f" - {c}: {n} imágenes")


In [ ]:
# Copiamos el MVP a disco LOCAL de la VM antes de entrenar. El montaje de Drive es un
# filesystem remoto (FUSE): leer miles de imágenes chicas directamente desde ahí durante
# el entrenamiento lo satura y termina desconectándose ("Transport endpoint is not
# connected"). Por eso entrenamos siempre desde disco local; Drive queda solo para el
# dataset "maestro" y los checkpoints/exports (escrituras poco frecuentes, que sí tolera bien).
LOCAL_MVP_DIR = '/content/data/mvp'

if os.path.isdir(LOCAL_MVP_DIR) and len(os.listdir(LOCAL_MVP_DIR)) > 0:
    print("Ya existe copia local, se omite:", LOCAL_MVP_DIR)
else:
    print("Copiando MVP a disco local (puede tardar unos minutos, una sola vez por sesión)...")
    shutil.copytree(MVP_DIR, LOCAL_MVP_DIR)

print(sorted(os.listdir(LOCAL_MVP_DIR)))


## 4. Pipeline de datos

Split 80/10/10 (train/val/test) usando `image_dataset_from_directory`, con augmentation ligero
en el set de entrenamiento (rotación, flip, zoom) para compensar el fondo controlado de
PlantVillage.


In [ ]:
IMG_SIZE = (224, 224)   # tamaño estándar de entrada para MobileNetV2
BATCH_SIZE = 32
SEED = 42

train_ds = tf.keras.utils.image_dataset_from_directory(
    LOCAL_MVP_DIR,
    validation_split=0.2,
    subset='training',
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
)

val_test_ds = tf.keras.utils.image_dataset_from_directory(
    LOCAL_MVP_DIR,
    validation_split=0.2,
    subset='validation',
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
)

class_names = train_ds.class_names
num_classes = len(class_names)
print("Clases:", num_classes)

# Partimos val_test_ds a la mitad: la mitad queda como validación (para early stopping),
# la otra mitad como test final que NO se toca hasta la evaluación de la sección 6.
val_batches = tf.data.experimental.cardinality(val_test_ds)
test_ds = val_test_ds.take(val_batches // 2)
val_ds = val_test_ds.skip(val_batches // 2)

print("Batches -> train:", tf.data.experimental.cardinality(train_ds).numpy(),
      "val:", tf.data.experimental.cardinality(val_ds).numpy(),
      "test:", tf.data.experimental.cardinality(test_ds).numpy())


In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.1),
], name='data_augmentation')

# Preprocesamiento específico de MobileNetV2 (escala los píxeles a [-1, 1]).
preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input

# Cache en disco LOCAL de la VM de Colab (no en RAM): el dataset decodificado completo
# no cabe en los ~12GB de RAM del runtime gratuito, lo que causaba el crash por falta de
# memoria a mitad del entrenamiento. /content es el disco local de la VM (rápido y con
# mucho más espacio que la RAM); se pierde al reiniciar el runtime, lo cual está bien
# porque es solo un caché reconstruible.
CACHE_DIR = '/content/tfcache'
os.makedirs(CACHE_DIR, exist_ok=True)

def prepare(ds, training=False, cache_name=None):
    if cache_name:
        ds = ds.cache(os.path.join(CACHE_DIR, cache_name))
    if training:
        ds = ds.map(lambda x, y: (data_augmentation(x, training=True), y), num_parallel_calls=AUTOTUNE)
    ds = ds.map(lambda x, y: (preprocess_input(x), y), num_parallel_calls=AUTOTUNE)
    return ds.prefetch(buffer_size=AUTOTUNE)

train_ds_p = prepare(train_ds, training=True, cache_name='train')
val_ds_p = prepare(val_ds, cache_name='val')
test_ds_p = prepare(test_ds, cache_name='test')


## 5. Modelo — MobileNetV2 (freeze → fine-tune)

**Fase 1:** base preentrenada en ImageNet, congelada; solo se entrena el clasificador nuevo.
**Fase 2:** se descongelan las últimas capas de la base y se afina con un learning rate bajo.


In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights='imagenet',
)
base_model.trainable = False  # Fase 1: congelada

inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(num_classes, activation='softmax')(x)
model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

model.summary()


In [ ]:
EPOCHS_HEAD = 10

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint(
        os.path.join(CKPT_DIR, 'head_best.keras'),
        monitor='val_accuracy', save_best_only=True,
    ),
]

history_head = model.fit(
    train_ds_p,
    validation_data=val_ds_p,
    epochs=EPOCHS_HEAD,
    callbacks=callbacks,
)


In [ ]:
# Fase 2: fine-tuning. Descongelamos solo el último bloque de la base
# (no toda la red, para no perder las features generales de ImageNet ni sobreajustar).
base_model.trainable = True

FINE_TUNE_AT = len(base_model.layers) - 30  # últimas ~30 capas se afinan
for layer in base_model.layers[:FINE_TUNE_AT]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),  # LR mucho más bajo que en fase 1
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

model.summary()


In [ ]:
EPOCHS_FINE_TUNE = 10
TOTAL_EPOCHS = EPOCHS_HEAD + EPOCHS_FINE_TUNE

callbacks_ft = [
    tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=4, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint(
        os.path.join(CKPT_DIR, 'finetuned_best.keras'),
        monitor='val_accuracy', save_best_only=True,
    ),
]

history_fine = model.fit(
    train_ds_p,
    validation_data=val_ds_p,
    epochs=TOTAL_EPOCHS,
    initial_epoch=history_head.epoch[-1] + 1,
    callbacks=callbacks_ft,
)


## 6. Evaluación con el split de prueba (laboratorio)

In [ ]:
test_loss, test_acc = model.evaluate(test_ds_p)
print(f"Test accuracy (PlantVillage, condiciones de laboratorio): {test_acc:.4f}")


## 7. Validación con PlantDoc (fotos de campo real)

Esta es la validación que de verdad importa para el producto: PlantDoc son fotos tomadas en
campo, no en laboratorio, y por eso el accuracy aquí va a ser más bajo que en el punto 6. Esa
brecha es exactamente el riesgo que el proyecto ya tiene documentado — **este es el número que
se reporta en la presentación**, no el del punto 6.

### Qué descargar

PlantDoc está en Roboflow: `universe.roboflow.com/joseph-nelson/plantdoc`, licencia CC BY 4.0
(uso comercial permitido, con la cita del paper de Singh et al. 2019).

**Elige el formato "Folder Structure" (classification).** Es un dataset pensado para detección de
objetos — 2.569 imágenes con 8.851 cajas delimitadoras — así que los formatos COCO, Pascal VOC y
YOLO te darán anotaciones, no carpetas por clase. Solo el export de clasificación sirve aquí.

Descomprímelo en Colab (por ejemplo en `/content/plantdoc_raw`) y apunta ahí `PLANTDOC_DESCARGA`
en la celda 7.1.

### Qué hace la celda 7.1

No tienes que elegir carpetas a mano. PlantDoc trae 30 clases de 13 especies con nombres propios
(`Tomato leaf late blight`), y tu modelo entrenó con los de PlantVillage
(`Tomato___Late_blight`). La celda traduce unos a otros, descarta las especies que no entrenaste
—manzana, uva, fresa, soya…— y deja `data/plantdoc/` con la estructura correcta.

De las 30 clases de PlantDoc, **15 corresponden a tus cultivos**. Dos clases del modelo se quedan
sin datos de campo porque PlantDoc no las cubre (`Corn_(maize)___healthy` y
`Tomato___Target_Spot`); la celda te dirá cuáles son.

Si alguna carpeta de maíz, tomate o papa aparece en la lista de descartadas, es que esa versión
del dataset la nombra distinto: añade el nombre a `MAPEO_PLANTDOC` y vuelve a ejecutar.

### Por qué la evaluación no usa `model.evaluate` a secas

Porque con menos clases que el modelo, los índices no coinciden: la clase 3 de PlantDoc puede ser
la 7 del modelo, y la comparación daría un accuracy bajísimo sin error alguno. La celda traduce
los índices antes de comparar, y además saca el accuracy por clase y el top-3 — que es lo que la
app enseña en la pantalla de Resultado.


In [ ]:
# ---------------------------------------------------------------------------
# 7.1 — Preparar PlantDoc: quedarte con tus cultivos y renombrar al esquema del modelo
#
# PlantDoc trae 30 clases de 13 especies y usa sus propios nombres
# ("Tomato leaf late blight"). Tu modelo entrenó con los de PlantVillage
# ("Tomato___Late_blight") y solo 3 especies. Esta celda hace la traducción,
# descarta lo que no te sirve y deja data/plantdoc/ listo.
#
# ANTES DE EJECUTAR: descarga PlantDoc de Roboflow eligiendo el formato
# "Folder Structure" (classification), NO uno de detección — los formatos COCO,
# VOC o YOLO te dan cajas delimitadoras, no carpetas por clase. Descomprímelo y
# apunta PLANTDOC_DESCARGA a esa carpeta.
#
# Roboflow lo parte en train/valid/test. Da igual: aquí no entrenamos con
# PlantDoc, solo evaluamos, así que se fusionan los tres.
# ---------------------------------------------------------------------------

PLANTDOC_DESCARGA = '/content/plantdoc_raw'   # <-- ajusta a donde descomprimiste

MAPEO_PLANTDOC = {
    # maíz
    'corn gray leaf spot':                  'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot',
    'corn leaf blight':                     'Corn_(maize)___Northern_Leaf_Blight',
    'corn rust leaf':                       'Corn_(maize)___Common_rust_',
    # papa
    'potato leaf':                          'Potato___healthy',
    'potato leaf early blight':             'Potato___Early_blight',
    'potato leaf late blight':              'Potato___Late_blight',
    # tomate
    'tomato leaf':                          'Tomato___healthy',
    'tomato early blight leaf':             'Tomato___Early_blight',
    'tomato septoria leaf spot':            'Tomato___Septoria_leaf_spot',
    'tomato leaf bacterial spot':           'Tomato___Bacterial_spot',
    'tomato leaf late blight':              'Tomato___Late_blight',
    'tomato leaf mosaic virus':             'Tomato___Tomato_mosaic_virus',
    'tomato leaf yellow virus':             'Tomato___Tomato_Yellow_Leaf_Curl_Virus',
    'tomato mold leaf':                     'Tomato___Leaf_Mold',
    'tomato two spotted spider mites leaf': 'Tomato___Spider_mites Two-spotted_spider_mite',
}

import re
import glob

def _norm(s):
    """Tolera mayúsculas, guiones y espacios de más entre versiones del dataset."""
    return re.sub(r'[\s_\-]+', ' ', s.strip().lower())

MAPEO_NORM = {_norm(k): v for k, v in MAPEO_PLANTDOC.items()}
EXTENSIONES = ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG')

assert os.path.isdir(PLANTDOC_DESCARGA), (
    f"No existe {PLANTDOC_DESCARGA}.\n"
    "Descomprime ahí el PlantDoc de Roboflow, o corrige PLANTDOC_DESCARGA."
)

# Toda carpeta que contenga imágenes cuenta como una clase, venga de train/,
# valid/ o test/. Si una clase aparece en varias, se acumulan.
origen_por_clase = {}
for raiz, _dirs, _archivos in os.walk(PLANTDOC_DESCARGA):
    imagenes = []
    for ext in EXTENSIONES:
        imagenes.extend(glob.glob(os.path.join(raiz, ext)))
    if imagenes:
        origen_por_clase.setdefault(os.path.basename(raiz), []).extend(imagenes)

print(f"{len(origen_por_clase)} carpetas con imágenes encontradas en la descarga.\n")

# Empezamos limpio para no mezclar con intentos anteriores.
if os.path.isdir(PLANTDOC_DIR):
    shutil.rmtree(PLANTDOC_DIR)
os.makedirs(PLANTDOC_DIR, exist_ok=True)

copiadas = {}
descartadas = []

for carpeta, imagenes in sorted(origen_por_clase.items()):
    clase_modelo = MAPEO_NORM.get(_norm(carpeta))
    if clase_modelo is None:
        descartadas.append((carpeta, len(imagenes)))
        continue
    destino = os.path.join(PLANTDOC_DIR, clase_modelo)
    os.makedirs(destino, exist_ok=True)
    ya = len(os.listdir(destino))
    for i, src in enumerate(imagenes):
        shutil.copy2(src, os.path.join(destino, f"{ya + i:04d}_{os.path.basename(src)}"))
    copiadas[clase_modelo] = copiadas.get(clase_modelo, 0) + len(imagenes)

print(f"{len(copiadas)} clases preparadas en {PLANTDOC_DIR}:")
for c in sorted(copiadas):
    print(f"  {copiadas[c]:4d}  {c}")

if descartadas:
    print(f"\n{len(descartadas)} carpetas descartadas (otras especies, o nombre no reconocido):")
    for c, n in descartadas:
        print(f"  {n:4d}  {c}")
    print("\nSi ves ahí arriba alguna de maíz, tomate o papa, es que su nombre no")
    print("coincide con MAPEO_PLANTDOC. Añádelo al diccionario y vuelve a ejecutar.")

# Clases del modelo que PlantDoc no cubre: no se evalúan, y conviene decirlo
# al reportar el número.
try:
    sin_datos = [c for c in class_names if c not in copiadas]
    if sin_datos:
        print(f"\n{len(sin_datos)} clases del modelo SIN datos de campo (quedan fuera de la evaluación):")
        for c in sin_datos:
            print("  -", c)
except NameError:
    print("\n(Ejecuta antes la sección 4 para comparar contra class_names.)")


In [ ]:
if os.path.isdir(PLANTDOC_DIR) and len(os.listdir(PLANTDOC_DIR)) > 0:
    # Igual que con MVP_DIR: copiamos a disco local antes de leer con tf.data, para no
    # depender del montaje de Drive durante la evaluación.
    LOCAL_PLANTDOC_DIR = '/content/data/plantdoc'
    if os.path.isdir(LOCAL_PLANTDOC_DIR):
        shutil.rmtree(LOCAL_PLANTDOC_DIR)
    shutil.copytree(PLANTDOC_DIR, LOCAL_PLANTDOC_DIR)

    plantdoc_ds = tf.keras.utils.image_dataset_from_directory(
        LOCAL_PLANTDOC_DIR,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        shuffle=False,
    )
    clases_plantdoc = plantdoc_ds.class_names

    # ----------------------------------------------------------------------
    # El detalle que hace que este número signifique algo.
    #
    # PlantDoc no cubre todas las clases del modelo, así que aquí hay menos
    # carpetas que en el entrenamiento y `image_dataset_from_directory` numera
    # las suyas de 0 en adelante por orden alfabético. Esos índices NO son los
    # del modelo: la clase 3 de PlantDoc puede ser la 7 del modelo.
    #
    # Si se llamara a model.evaluate() directamente, compararía índice contra
    # índice y daría un accuracy bajísimo sin ningún error — pareciendo que el
    # modelo no generaliza, cuando en realidad estaríamos comparando tomate
    # contra maíz. Por eso traducimos los índices antes de comparar.
    # ----------------------------------------------------------------------
    desconocidas = [c for c in clases_plantdoc if c not in class_names]
    assert not desconocidas, (
        f"Estas carpetas no son clases del modelo: {desconocidas}\n"
        "Revisa MAPEO_PLANTDOC en la celda 7.1."
    )
    a_indice_modelo = [class_names.index(c) for c in clases_plantdoc]

    plantdoc_ds_p = plantdoc_ds.map(
        lambda x, y: (preprocess_input(x), y), num_parallel_calls=AUTOTUNE
    )

    # shuffle=False en ambos, así que el orden coincide.
    y_plantdoc = np.concatenate([y.numpy() for _, y in plantdoc_ds])
    y_real = np.array([a_indice_modelo[i] for i in y_plantdoc])

    probabilidades = model.predict(plantdoc_ds_p, verbose=0)
    y_predicho = probabilidades.argmax(axis=1)

    pd_acc = float((y_predicho == y_real).mean())

    # Top-3, porque es lo que la app enseña de verdad en la pantalla de Resultado.
    top3 = np.argsort(probabilidades, axis=1)[:, -3:]
    pd_top3 = float(np.mean([y_real[i] in top3[i] for i in range(len(y_real))]))

    print(f"\nImágenes de campo evaluadas: {len(y_real)}")
    print(f"Clases con datos de campo:   {len(clases_plantdoc)} de {len(class_names)} del modelo")
    print(f"\nAccuracy en PlantDoc (campo real): {pd_acc:.4f}")
    print(f"Accuracy top-3 (lo que ve el usuario): {pd_top3:.4f}")
    print(f"Brecha laboratorio -> campo: {test_acc - pd_acc:+.4f}")

    # Por clase: casi siempre es más útil que el número global, porque suele
    # haber dos o tres clases que arrastran toda la caída.
    print("\nAccuracy por clase:")
    for nombre in clases_plantdoc:
        idx = class_names.index(nombre)
        mascara = y_real == idx
        n = int(mascara.sum())
        if n:
            acc_c = float((y_predicho[mascara] == idx).mean())
            print(f"  {acc_c:6.1%}  ({n:3d} fotos)  {nombre}")

    print(
        "\nAl reportarlo, di sobre qué se midió: "
        f"{len(y_real)} fotos de campo en {len(clases_plantdoc)} de las "
        f"{len(class_names)} clases del modelo."
    )
else:
    print(f"No se encontró PlantDoc en {PLANTDOC_DIR}.")
    print("Ejecuta antes la celda 7.1, que prepara las carpetas a partir de la descarga de Roboflow.")
    print("Sin esto, el número del punto 6 puede ser engañosamente optimista para el uso real en campo.")


## 8. Exportar a TFLite cuantizado + labels

Exportamos con cuantización dinámica (int8 de pesos, buen balance tamaño/precisión para
on-device) y el archivo de labels en el mismo orden que usa el modelo, listo para copiarse a
`assets/` en el proyecto Android.


In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]  # cuantización dinámica post-entrenamiento
tflite_model = converter.convert()

tflite_path = os.path.join(EXPORT_DIR, 'agrosense_crop_model.tflite')
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)

labels_path = os.path.join(EXPORT_DIR, 'labels.txt')
with open(labels_path, 'w') as f:
    f.write('\n'.join(class_names))

size_mb = os.path.getsize(tflite_path) / (1024 * 1024)
print(f"Modelo exportado: {tflite_path} ({size_mb:.2f} MB)")
print(f"Labels exportadas: {labels_path}")
print(f"\n{len(class_names)} clases, en este orden (debe coincidir con la salida del modelo):")
for i, c in enumerate(class_names):
    print(f"  {i}: {c}")


### 8.2 Incrustar la metadata (obligatorio para la app)

El `.tflite` que acaba de salir **todavía no le sirve a la app Android**. El proyecto Kotlin usa
`org.tensorflow:tensorflow-lite-task-vision`, y la Task Library **rechaza** los modelos que no
llevan *TFLite Model Metadata* embebida: necesita saber cómo normalizar la imagen de entrada y
de dónde sacar las etiquetas. `TFLiteConverter` no escribe nada de eso.

La celda siguiente toma el modelo exportado y le incrusta:

- **La normalización de entrada.** `mobilenet_v2.preprocess_input` escala los píxeles a `[-1, 1]`,
  lo que en la convención de la metadata es `mean = 127.5` y `std = 127.5`. Esos números tienen
  que corresponderse con el preprocesamiento de la sección 4 — si algún día cambias de backbone,
  cámbialos aquí también o el modelo recibirá imágenes en un rango que no espera.
- **El archivo de etiquetas**, con el mismo orden de `class_names`, dentro del propio `.tflite`.

Que las etiquetas viajen dentro del modelo es justo lo que cierra el riesgo del orden: el
`ImageClassifier` de la app prefiere la etiqueta embebida y solo cae al `labels.txt` suelto
cuando el modelo no trae ninguna.


In [ ]:
!pip install -q tflite-support==0.4.4

from tflite_support.metadata_writers import image_classifier, writer_utils

# Tiene que coincidir con el preprocess_input de la sección 4:
# mobilenet_v2 escala a [-1, 1], que en la metadata se expresa así.
NORM_MEAN = 127.5
NORM_STD = 127.5

MODELO_CON_METADATA = os.path.join(EXPORT_DIR, 'agrosense_crop_model_meta.tflite')

writer = image_classifier.MetadataWriter.create_for_inference(
    writer_utils.load_file(tflite_path),
    input_norm_mean=[NORM_MEAN],
    input_norm_std=[NORM_STD],
    label_file_paths=[labels_path],
)
writer_utils.save_file(writer.populate(), MODELO_CON_METADATA)

# Verificación: releemos la metadata del archivo final y comprobamos que las
# etiquetas que quedaron dentro son exactamente las del modelo, en el mismo
# orden. Si esto falla, NO copies el modelo a la app.
from tflite_support import metadata as _metadata

desplazador = _metadata.MetadataDisplayer.with_model_file(MODELO_CON_METADATA)
etiquetas_incrustadas = desplazador.get_packed_associated_file_list()
contenido = desplazador.get_associated_file_buffer(etiquetas_incrustadas[0]).decode()
leidas = [l for l in contenido.splitlines() if l.strip()]

assert leidas == class_names, (
    "Las etiquetas incrustadas no coinciden con class_names.\n"
    f"  incrustadas: {leidas[:5]}...\n"
    f"  esperadas:   {class_names[:5]}..."
)

size_mb = os.path.getsize(MODELO_CON_METADATA) / (1024 * 1024)
print(f"OK — modelo con metadata: {MODELO_CON_METADATA} ({size_mb:.2f} MB)")
print(f"{len(leidas)} etiquetas incrustadas, en el orden correcto.")
print("\nEste es el archivo que va a la app (renómbralo a agrosense_crop_model.tflite).")


### 8.3 Siguiente paso en Android Studio

El proyecto Kotlin ya está listo para recibir el modelo: la migración de la UI dejó el
clasificador, la pantalla de captura y el guardado en Room funcionando, degradando con un aviso
mientras no haya `.tflite`. Solo faltan los dos archivos.

1. Descarga de Drive **`export/agrosense_crop_model_meta.tflite`** (el que produce la sección
   8.2) y `export/labels.txt`.

   Ojo: en esa misma carpeta está también `agrosense_crop_model.tflite`, el que escribe la
   sección 8 sin metadata. **Ese no sirve** — la Task Library lo rechaza al cargarlo.
2. **Renombra** el archivo `_meta` a `agrosense_crop_model.tflite` — es el nombre literal que
   espera `ImageClassifier.MODELO`.
3. Crea `app/src/main/assets/` en el proyecto y copia **los dos archivos juntos**.
4. Recompila:
   ```
   JAVA_HOME="/c/Program Files/Android/Android Studio/jbr" ./gradlew assembleDebug --no-daemon
   ```

La dependencia ya está en el catálogo de versiones (`tensorflow-lite-task-vision 0.4.4`), así que
no hay que tocar Gradle.

> ⚠️ **Los dos archivos viajan juntos, y `labels.txt` no se edita a mano.** El clasificador
> resuelve el nombre de cada predicción por índice: prefiere la etiqueta incrustada en el modelo
> y, si no la hay, cae al `labels.txt` de `assets/`. Un archivo desalineado con las salidas del
> modelo no produce ningún error — simplemente etiqueta mal todos los diagnósticos. Por eso la
> sección 8.2 verifica la correspondencia con un `assert` antes de darte el archivo por bueno.

> **Nota sobre los diagnósticos ya guardados.** El nombre de la clase se persiste en la tabla de
> diagnósticos en el momento de clasificar. Las filas creadas antes de que exista el modelo
> guardan el texto de respaldo (`Clase 3`) y no se renombran solas al añadirlo después.

Con esto queda cerrada la prueba de concepto del modelo de cultivos: foto → predicción, sin
login, sin Firestore — el hito de las semanas 1-2. El módulo ganadero reutiliza exactamente este
mismo notebook (secciones 2-8) cambiando solo el dataset de origen (Lumpy Skin Disease /
Roboflow) y las etiquetas de salida.
